# Breakout Hit Classifier

**Goal:** Predict whether a game will become a "breakout hit" using observable characteristics at the time of observation.  
**Dataset:** Pre-built `vg_revenue_prediction.csv` subset — 5.47M rows, 32 columns  
**Tools:** pandas · scikit-learn · XGBoost · matplotlib · seaborn

---

## What is a breakout hit?

In this dataset, `target_breakout_hit` is a binary label:
- `1` — the game achieved breakout success (explosive growth in players and revenue)
- `0` — the game did not

The class split is **86.7% not breakout / 13.3% breakout** — this is an imbalanced classification problem, which affects how we train and evaluate the model.

## Modeling approach

| Step | What we do |
|---|---|
| 1 | Load the pre-built subset and inspect the data |
| 2 | Sample down to 200k rows for laptop-friendly training |
| 3 | Encode categorical features and split into train/test sets |
| 4 | Train a Logistic Regression baseline and an XGBoost classifier |
| 5 | Identify and fix a feature leakage problem |
| 6 | Retrain the honest model and interpret feature importance |

## Why two models?

We always start with a **simple baseline** (Logistic Regression) before trying something complex. If a simple model already performs well, there's no reason to add complexity. If the complex model does much better, it tells us the relationship between features and target is non-linear — something a simple model can't capture.

---

## Step 1 — Load the data

We use the pre-built `vg_revenue_prediction.csv` subset rather than the full 5GB dataset. The dataset creator already selected the most relevant columns for revenue and breakout prediction tasks, giving us a focused 32-column dataset.

**Why kagglehub instead of a local path?**  
The raw data is gitignored (too large to commit). `kagglehub.dataset_download()` is idempotent — if the data is already cached locally it returns the path instantly without re-downloading. This makes the notebook reproducible on any machine without storing the data in the repo.

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────
DATASET_HANDLE   = 'amith1707/video-game-market-price-and-revenue-dataset'
SUBSET_RELATIVE  = Path(
    'data/subsets/videogames_revenue_prediction.csv/vg_revenue_prediction.csv'
)
SAMPLE_SIZE      = 200_000   # rows to sample — keeps training fast on a laptop
RANDOM_STATE     = 42        # fixed seed for reproducibility
TARGET_COL       = 'target_breakout_hit'

# ── Load ──────────────────────────────────────────────────────────────────────
dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
subset_csv   = dataset_path / SUBSET_RELATIVE

print(f'Loading: {subset_csv}')
df = pd.read_csv(subset_csv)

print(f'\nFull dataset shape: {df.shape}')
print(f'\nTarget distribution (full dataset):')
print(df[TARGET_COL].value_counts(normalize=True).round(3))
print(f'\nColumns: {list(df.columns)}')

---

## Step 2 — Sample and inspect

The full subset has 5.47M rows — training a model on all of them would take a long time on a laptop and isn't necessary. A random sample of 200,000 rows is large enough to train a reliable model while keeping iteration fast.

**What we check here:**
- **Shape** — confirms the sample size
- **Null counts** — missing values need to be handled before training; most ML models can't handle NaN
- **Target distribution** — confirms the class imbalance held up in the sample (should still be ~87/13)

In [ ]:
df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

print(f'Sampled shape: {df_sample.shape}')

print(f'\nNull counts (only columns with nulls):')
nulls = df_sample.isnull().sum()
null_cols = nulls[nulls > 0]
print(null_cols if len(null_cols) > 0 else '  None — clean dataset')

print(f'\nTarget distribution in sample:')
print(df_sample[TARGET_COL].value_counts(normalize=True).round(3))

---

## Step 3 — Feature engineering and train/test split

Before training, we need to:

**1. Drop columns that shouldn't be features:**
- `game_id`, `game_name` — identifier columns with no predictive value
- `publisher` — too high cardinality (thousands of unique values); we keep `publisher_tier` instead
- `genre_code` — redundant with `genre` (same information, different encoding)
- `target_revenue_next_4w` — this is another target variable, not a feature; using it would be **data leakage**

**2. Encode categorical columns:**  
ML models work with numbers, not strings. `LabelEncoder` converts each unique string to an integer (e.g. `"RPG"` → `3`, `"Action"` → `0`).

**3. Split into train and test sets:**  
We hold out 20% of the data as a test set — the model never sees this during training. `stratify=y` ensures the 87/13 class split is preserved in both the train and test sets.

**What is data leakage?**  
Leakage happens when a feature gives the model information it wouldn't have in a real prediction scenario. Including `target_revenue_next_4w` (a future measurement) as a feature would let the model "cheat" — it would look like a great predictor but would fail completely on real data where the future isn't known yet.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── Drop columns that shouldn't be features ───────────────────────────────────
DROP_COLS = [
    'game_id',              # identifier — no predictive value
    'game_name',            # identifier — no predictive value
    'publisher',            # too high cardinality; publisher_tier captures this more cleanly
    'genre_code',           # redundant with genre
    'target_revenue_next_4w' # another target variable — would be data leakage
]
df_model = df_sample.drop(columns=DROP_COLS)

# ── Encode categorical columns ────────────────────────────────────────────────
# LabelEncoder converts strings to integers so the model can process them
CAT_COLS = ['genre', 'platform_code', 'publisher_tier']
le = LabelEncoder()
for col in CAT_COLS:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# ── Separate features (X) from target (y) ────────────────────────────────────
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]

# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y preserves the 87/13 class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set : {X_train.shape}')
print(f'Test set     : {X_test.shape}')
print(f'\nFeatures used ({len(X.columns)}):')
for col in X.columns:
    print(f'  {col}')

---

## Step 4 — Train baseline and XGBoost models

### Logistic Regression (baseline)
A simple linear model that draws a straight line (or hyperplane in higher dimensions) to separate the two classes. Fast to train, easy to interpret, but can't capture complex non-linear relationships.

`class_weight='balanced'` tells the model to pay more attention to the minority class (breakout hits). Without this, a model trained on 87/13 imbalanced data will simply predict "not a breakout" for everything and still report 87% accuracy — which is completely useless.

### XGBoost
A gradient boosted tree model — one of the most widely used algorithms in industry for tabular data. It builds many small decision trees sequentially, each one correcting the mistakes of the previous one.

`scale_pos_weight` is XGBoost's version of class weighting — we set it to the ratio of negative to positive samples so the model treats breakout hits as more important during training.

### Evaluation metrics
- **Precision** — of all games the model predicted as breakout hits, what fraction actually were? (avoid false alarms)
- **Recall** — of all actual breakout hits, what fraction did the model catch? (avoid missing real hits)
- **F1-score** — harmonic mean of precision and recall; useful single number when classes are imbalanced
- **ROC-AUC** — measures how well the model ranks breakout hits above non-hits across all possible decision thresholds. 0.5 = random guessing, 1.0 = perfect

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# ── Logistic Regression baseline ──────────────────────────────────────────────
print('Training Logistic Regression (baseline)...')
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # compensate for 87/13 imbalance
    random_state=RANDOM_STATE
)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print('\n--- Logistic Regression ---')
print(classification_report(
    y_test, lr_preds,
    target_names=['Not Breakout', 'Breakout Hit']
))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────────────
# scale_pos_weight = ratio of negatives to positives — handles class imbalance
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

print('\nTraining XGBoost...')
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print('\n--- XGBoost (v1) ---')
print(classification_report(
    y_test, xgb_preds,
    target_names=['Not Breakout', 'Breakout Hit']
))
print(f'ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}')

---

## Step 5 — Investigate feature importance

Before trusting the XGBoost results, we look at what the model actually learned. Feature importance tells us which columns the model relied on most when making predictions.

**If one feature dominates with extremely high importance, that's a red flag.** It usually means one of two things:
1. That feature genuinely is the strongest predictor (rare for a single feature to be this dominant)
2. That feature is **leaking the answer** — it encodes information about the target that wouldn't be available in a real prediction scenario

We saw XGBoost perform very well. Let's find out why.

In [ ]:
feature_importance_v1 = pd.DataFrame({
    'feature':    X.columns,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=feature_importance_v1,
    x='importance', y='feature',
    color='steelblue', ax=ax
)
ax.set_title('XGBoost Feature Importance — v1 (with is_aaa)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(feature_importance_v1.to_string(index=False))

### Red flag — `is_aaa` dominates

The model assigned **86%+ of its importance to a single feature: `is_aaa`**. Everything else combined accounts for less than 14%.

This is **feature leakage**. In this simulation, the `is_aaa` flag (whether a game is a AAA/big-budget title) was almost certainly used directly when generating the `target_breakout_hit` labels — meaning the model didn't learn anything interesting about what makes a breakout hit. It just learned one rule: *"AAA games are breakout hits."*

**Why is this a problem in practice?**  
If you deployed this model, it would tell you nothing you didn't already know from looking at `is_aaa` directly. It adds zero value as a predictor. A good data scientist always asks: *"If a model looks too good, what is it actually learning?"*

**The fix:** Remove `is_aaa` and retrain. This forces the model to find genuine signals in the data rather than relying on one dominant shortcut.

---

## Step 6 — Retrain without `is_aaa` (the honest model)

By removing `is_aaa` we force the model to learn from the remaining 25 features. The result will be a more honest and interpretable model — one that tells us something genuine about what drives breakout success.

We expect the ROC-AUC to drop somewhat compared to v1, since we removed the most "powerful" (but leaky) feature. That drop is acceptable — it means the model is now working harder to find real signal rather than taking a shortcut.

In [ ]:
# ── Remove is_aaa and rebuild train/test sets ─────────────────────────────────
X_clean = X.drop(columns=['is_aaa'])

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_clean, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# ── Retrain XGBoost ───────────────────────────────────────────────────────────
print('Retraining XGBoost without is_aaa...')
xgb2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)
xgb2.fit(X_train2, y_train2)
xgb2_preds = xgb2.predict(X_test2)
xgb2_proba = xgb2.predict_proba(X_test2)[:, 1]

print('\n--- XGBoost v2 (without is_aaa) ---')
print(classification_report(
    y_test2, xgb2_preds,
    target_names=['Not Breakout', 'Breakout Hit']
))
print(f'ROC-AUC: {roc_auc_score(y_test2, xgb2_proba):.4f}')

---

## Step 7 — Feature importance of the honest model

With `is_aaa` removed, the importance scores are now distributed across many features — which is what we'd expect from a model that's genuinely learning patterns rather than relying on one shortcut.

The features at the top of this chart are the model's best answer to: **"What actually predicts whether a game becomes a breakout hit?"**

In [ ]:
feature_importance_v2 = pd.DataFrame({
    'feature':    X_clean.columns,
    'importance': xgb2.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=feature_importance_v2,
    x='importance', y='feature',
    color='steelblue', ax=ax
)
ax.set_title('XGBoost Feature Importance — v2 (without is_aaa)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(feature_importance_v2.to_string(index=False))

### What the honest model learned

| Feature | Importance | What it means |
|---|---|---|
| `publisher_avg_budget` | ~0.40 | Big marketing budgets drive visibility — breakout hits are partly manufactured |
| `is_multiplayer` | ~0.10 | Multiplayer games have built-in virality — one player brings their friends |
| `concurrent_players` | ~0.08 | Early traction predicts future traction — momentum compounds |
| `hype_score` | ~0.07 | Pre-release momentum matters — games that generate buzz before launch tend to maintain it |
| `twitch_viewers_proxy` | ~0.06 | Streamer pickup drives explosive growth — this is how many modern breakouts happen |

**The business insight:** Breakout hits are largely structurally determined — by publisher resources and multiplayer design — rather than being random. Early engagement signals (hype, Twitch views, concurrent players) are the best real-time indicators once a game is live. This means publishers can identify likely breakout candidates early and double down on marketing spend before the breakout fully materialises.

---

## Summary

| Model | ROC-AUC | Notes |
|---|---|---|
| Logistic Regression (baseline) | 0.8562 | Simple, interpretable, decent baseline |
| XGBoost v1 | ~0.99 | Suspiciously high — investigate |
| XGBoost v2 (final) | 0.9890 | Honest model after removing leaky feature |

### Key lessons from this notebook

**1. Always inspect feature importance before trusting model results.**  
A model that looks too good usually is. `is_aaa` scoring 86% importance was the signal that something was wrong.

**2. Feature leakage is subtle and common.**  
Leakage doesn't always mean including the target column directly. It can mean including a feature that was generated using the target, or one that wouldn't be available at prediction time. Always think about *what information the model would realistically have* when making a real prediction.

**3. Class imbalance requires deliberate handling.**  
With 87/13 split, a model that predicts "not a breakout" for everything would be 87% accurate — and completely useless. `class_weight='balanced'` and `scale_pos_weight` are the tools that fix this.

**4. ROC-AUC is more useful than accuracy for imbalanced problems.**  
Accuracy is misleading when one class dominates. ROC-AUC measures ranking quality across all thresholds — a much more honest picture of model performance.

**5. A simpler story is often a better story.**  
The v2 model has slightly lower raw performance than v1, but it tells a genuine and actionable story about what drives breakout success. That's more valuable than a high number that doesn't mean anything.